# CS781 Phase 5 — Experiment 2: Model4 + LoRMA+ (Fast)

**Task:** Legal Statute Identification (100-label multilabel classification)  
**Phase-1 foundation:** Model4 — 17-layer efficient decoder with GQA, RoPE, RMSNorm and SwiGLU  
**Technique:** **LoRMA+ — Low-Rank Multiplicative Adaptation with additive identity rank inflation**

This notebook is intentionally lighter than Experiment 1:

- **3 document views:** HEAD / statute-EVIDENCE / TAIL
- **320 tokens/view**
- bottom **9 blocks frozen and executed under `no_grad`**
- LoRMA+ adapters only in the **top 8 blocks**
- Q/K/V/O attention projections adapted (**v2: + SwiGLU gate/up/down, switchable**)
- **2 epochs**
- **1 DEV evaluation per epoch**
- cached tokenization
- FP16 + TF32
- exact Phase-5 evaluator-compatible output
- no expensive TRAIN+DEV refit or heldout/refit ensemble by default

### LoRMA+ implemented here

For frozen pretrained weights \(W_0\):

\[
h = \left(I + \frac{\alpha}{r}BA\right)W_0x
\]

and, using associativity,

\[
h = W_0x + \frac{\alpha}{r}B(A(W_0x)).
\]

`A` is randomly initialized and `B` is initialized to zero. Therefore the injected network begins **exactly equal to your Phase-1 Model4 checkpoint**.

Reference: Bihany, Patel & Modi (2025), *LoRMA: Low-Rank Multiplicative Adaptation for LLMs*, Findings of ACL 2025, DOI: 10.18653/v1/2025.findings-acl.527.

### v2 tweaks (LoRMA+ equation, frozen prefix, views, loss, EMA, threshold CV all unchanged)
1. rank 8 -> **16** (alpha 16 -> 32, same scaling = 2) and adapters also on the FFN projections (`LORMA_ADAPT_FFN`, one-line switch) -> more capacity for ~+3% compute.
2. LoRMA LR 3e-4 -> **5e-4**, head LR 5e-4 -> **8e-4** (adapters + zero-init B need a larger LR to move in only 2 epochs).
3. Micro-batch 2 -> **4** (accum 8 -> 4, effective batch still 16), dev batch 10 -> 16: better T4 utilisation, shorter epochs.
4. `persistent_workers` only on the TRAIN loader, so an interrupt can no longer break the DEV/TEST loaders.
5. Optional last cell blends with Exp 1 probabilities (weight chosen by cross-fitted DEV F1) and writes an extra submission ZIP.


### Submission outputs in this version

The full run now creates **three core Exp-2 candidate ZIPs**:

1. **Heldout LoRMA+** — selected/calibrated on clean DEV.
2. **Fresh TRAIN+DEV refit** — starts again from the original Phase-1 Model4 checkpoint.
3. **Heldout + refit probability ensemble** — 50/50 probability average using heldout-selected thresholds.

The refit does **not** run repeated DEV evaluation, so the extra stage is much cheaper than repeating the whole heldout experiment.  
If Experiment-1 probability files are also uploaded, the final optional cell can still create an additional **Exp1 + Exp2** blend submission.


## Kaggle inputs

Attach:

1. your Phase-1 Model4 **`eff_full_best*.pt`**
2. its exact **`tokenizer.json`**

The notebook searches `/kaggle/input` automatically. If needed, set the two override variables in Cell 1.


In [1]:
# CELL 1 — imports, reproducibility, fast Exp-2 configuration
import os, re, gc, json, math, time, random, hashlib, zipfile, shutil, heapq, contextlib, subprocess, sys, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from huggingface_hub import hf_hub_download, login

SEED = 1234
NOTEBOOK_VERSION = "exp2_model4_lormaplus_fast_v4_refit_ensemble"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/phase5_exp2_model4_lorma_fast_v4")
CACHE_DIR = ROOT / "cache"
ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Exact Phase-1 Model4 constants.
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
FFN_DIM = 2048
DROPOUT = 0.0
MAX_POSITIONS = 2560
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5

# Phase-5.
N_LABELS = 100

# Faster long-document pipeline than Exp 1.
N_CHUNKS = 3
CHUNK_LEN = 320
CHAR_WINDOW = 5000
VIEW_NAMES = ["head", "statute_evidence", "tail"]

# Frozen prefix + LoRMA top section.
ADAPT_START_BLOCK = 9
ADAPTED_BLOCKS = list(range(ADAPT_START_BLOCK, N_LAYERS))
LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj")
LORMA_ADAPT_FFN = True                                   # v2: also adapt SwiGLU projections (set False for attention-only)
LORMA_FF_TARGETS = ("gate_proj", "up_proj", "down_proj")
LORMA_RANK = 16                                          # v2: was 8
LORMA_ALPHA = 32.0                                       # v2: was 16 (alpha/r = 2, unchanged)
LORMA_INIT_STD = 0.02

# Training.
MICRO_BATCH_SIZE = 4          # v2: was 2 (effective batch 4*4 = 16 unchanged). OOM -> 2 / 8
DEV_BATCH_SIZE = 16           # v2: was 10
GRAD_ACCUM_STEPS = 4          # v2: was 8
EPOCHS = 2
LORMA_LR = 5e-4               # v2: was 3e-4
HEAD_LR = 8e-4                # v2: was 5e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EMA_DECAY = 0.998
GRAD_CLIP = 1.0
USE_AMP = True
NUM_WORKERS = 2

# Safe with RIGHT padding + causal attention; valid tokens never see later padding.
FAST_CAUSAL_NO_MASK = True

# Threshold tuning.
THR_GRID = np.round(np.arange(0.02, 0.8001, 0.01), 3)
SHRINK_TAUS = [25.0, 50.0, 100.0, 200.0]
CV_REPEATS = 2

MODEL4_CKPT_OVERRIDE = None
TOKENIZER_OVERRIDE = None

def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def norm_text(s):
    return re.sub(r"\s+"," ",str(s).lower()).strip()

def text_hash(s):
    return hashlib.sha256(norm_text(s).encode("utf-8")).hexdigest()

def atomic_torch_save(obj, path):
    path=Path(path); tmp=Path(str(path)+".tmp")
    torch.save(obj,tmp); os.replace(tmp,path)

print("ROOT:",ROOT)
print("Adapted blocks:",ADAPTED_BLOCKS)
print("LoRMA targets:",LORMA_TARGETS,"+ FFN" if LORMA_ADAPT_FFN else "", LORMA_FF_TARGETS if LORMA_ADAPT_FFN else "")
print("Views:",VIEW_NAMES,"chunk_len:",CHUNK_LEN)


Device: cuda
GPU: Tesla T4
ROOT: /kaggle/working/phase5_exp2_model4_lorma_fast_v4
Adapted blocks: [9, 10, 11, 12, 13, 14, 15, 16]
LoRMA targets: ('q_proj', 'k_proj', 'v_proj', 'o_proj') + FFN ('gate_proj', 'up_proj', 'down_proj')
Views: ['head', 'statute_evidence', 'tail'] chunk_len: 320


In [2]:
# CELL 2 — find Phase-1 Model4 checkpoint + exact tokenizer
all_files=[]
for base in ["/kaggle/input","/kaggle/working"]:
    if os.path.exists(base):
        for root,dirs,files in os.walk(base):
            for f in files:
                all_files.append(os.path.join(root,f))

ckpt_cands=[p for p in all_files if p.lower().endswith(".pt") and "eff_full_best" in os.path.basename(p).lower()]
tok_cands=[p for p in all_files if os.path.basename(p).lower()=="tokenizer.json"]

def asset_score(p,words):
    s=p.lower()
    return sum(10 for w in words if w in s)-len(p)/100000.0

if MODEL4_CKPT_OVERRIDE:
    MODEL4_CKPT=str(MODEL4_CKPT_OVERRIDE)
else:
    assert ckpt_cands,"Could not find eff_full_best*.pt. Attach the Phase-1 Model4 best checkpoint."
    MODEL4_CKPT=sorted(
        ckpt_cands,
        key=lambda p:asset_score(p,["model4","decoder-efficient","phase1","efficient"]),
        reverse=True
    )[0]

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE=str(TOKENIZER_OVERRIDE)
else:
    assert tok_cands,"Could not find tokenizer.json. Attach the exact Phase-1 Model4 tokenizer."
    ckpt_dir=os.path.dirname(MODEL4_CKPT).lower()
    TOKENIZER_FILE=sorted(
        tok_cands,
        key=lambda p:(
            20 if os.path.dirname(p).lower()==ckpt_dir else 0,
            asset_score(p,["model4","decoder-efficient","phase1"])
        ),
        reverse=True
    )[0]

print("SELECTED MODEL4:",MODEL4_CKPT)
print("SELECTED TOKENIZER:",TOKENIZER_FILE)
print("Model4 SHA256:",sha256_file(MODEL4_CKPT))
print("Tokenizer SHA256:",sha256_file(TOKENIZER_FILE))


SELECTED MODEL4: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
SELECTED TOKENIZER: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Model4 SHA256: 529cfef4605822c82075aee8254259ea8a42b91ec76ca41e32bd99ea7b61ed74
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact Phase-1 LegalBPE tokenizer class
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:",len(tokenizer))
print("special ids:",tokenizer.pad_token_id,tokenizer.unk_token_id,tokenizer.bos_token_id,tokenizer.eos_token_id)
assert len(tokenizer)==VOCAB_SIZE
assert tokenizer.pad_token_id==0
print("✅ tokenizer sanity PASS")


Tokenizer class ready.
Tokenizer vocab: 24576
special ids: 0 1 2 3
✅ tokenizer sanity PASS


In [4]:
# CELL 4 — authenticate/download official Phase-5 data
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret unavailable:",repr(e)[:120])

if HF_TOKEN:
    login(token=HF_TOKEN,add_to_git_credential=False)

HF_REPO="Exploration-Lab/CS-781-Capstone"
FNS={
    "train":"Phase_5_LSI/Train/train.json",
    "dev":"Phase_5_LSI/Train/dev.json",
    "test":"Phase_5_LSI/Test/test.json",
}
paths={
    k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN)
    for k,v in FNS.items()
}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])

print("rows:",len(train_raw),len(dev_raw),len(test_raw))
assert len(train_raw)==42750 and len(dev_raw)==10181 and len(test_raw)==13019

for name,recs,labeled in [("TRAIN",train_raw,True),("DEV",dev_raw,True),("TEST",test_raw,False)]:
    ids=[str(r["id"]) for r in recs]
    assert len(ids)==len(set(ids)),f"{name} duplicate IDs"
    for r in recs:
        assert isinstance(r.get("text"),str)
        if labeled:
            assert isinstance(r.get("labels"),list)
            r["labels"]=sorted(set(int(x) for x in r["labels"]))
            assert all(0<=x<N_LABELS for x in r["labels"])

assert not any("labels" in r for r in test_raw)
print("✅ schema/ID/label sanity PASS")


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

rows: 42750 10181 13019
✅ schema/ID/label sanity PASS


In [5]:
# CELL 5 — leakage-aware clean DEV + class weights
train_hashes={text_hash(r["text"]) for r in train_raw}
dev_hashes={text_hash(r["text"]) for r in dev_raw}
test_hashes={text_hash(r["text"]) for r in test_raw}

print("text overlap train-dev:",len(train_hashes&dev_hashes))
print("text overlap train-test:",len(train_hashes&test_hashes))
print("text overlap dev-test:",len(dev_hashes&test_hashes))

CLEAN_IDX=np.asarray(
    [i for i,r in enumerate(dev_raw) if text_hash(r["text"]) not in train_hashes],
    dtype=np.int64
)
clean_dev=[dev_raw[i] for i in CLEAN_IDX]
train_records=train_raw
test_records=test_raw

print("official DEV:",len(dev_raw),"clean DEV:",len(clean_dev),"removed:",len(dev_raw)-len(clean_dev))
assert len(clean_dev)>9000

train_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in train_records: train_counts[r["labels"]]+=1

dev_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in clean_dev: dev_counts[r["labels"]]+=1

assert np.all(train_counts>0)
assert np.all(dev_counts>0)

raw_pw=(len(train_records)-train_counts)/train_counts
POS_WEIGHT=np.clip(np.sqrt(raw_pw),1.0,10.0).astype(np.float32)
print("pos_weight min/median/max:",POS_WEIGHT.min(),np.median(POS_WEIGHT),POS_WEIGHT.max())

official_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in dev_raw]
clean_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in clean_dev]
json.dump(official_truth,open(ROOT/"dev_truth_official.json","w"))
json.dump(clean_truth,open(ROOT/"dev_truth_clean.json","w"))


text overlap train-dev: 81
text overlap train-test: 109
text overlap dev-test: 62
official DEV: 10181 clean DEV: 10063 removed: 118
pos_weight min/median/max: 1.6261922 7.871644 10.0


In [6]:
# CELL 6 — fast 3-view statute-aware token cache
CUE_RE=re.compile(
    r"\b(section|sections|sec\.?|article|articles|act|acts|rule|rules|"
    r"provision|provisions|ipc|crpc|constitution)\b",re.I
)

def _char_window(text,center,width=CHAR_WINDOW):
    if len(text)<=width: return text
    st=max(0,min(len(text)-width,int(center-width//2)))
    return text[st:st+width]

def select_views(text):
    n=len(text)
    if n<=CHAR_WINDOW:
        return [text,text,text]

    head=text[:CHAR_WINDOW]
    tail=text[-CHAR_WINDOW:]

    hits=[m.start() for m in CUE_RE.finditer(text)]
    if hits:
        best_i=best_j=0; j=0
        for i in range(len(hits)):
            if j<i: j=i
            while j+1<len(hits) and hits[j+1]-hits[i]<=CHAR_WINDOW:
                j+=1
            if j-i>best_j-best_i:
                best_i,best_j=i,j
        center=(hits[best_i]+hits[best_j])//2
        evid=_char_window(text,center)
    else:
        evid=_char_window(text,n//2)

    return [head,evid,tail]

def encode_view(text,view_idx):
    ids=tokenizer.encode(text)
    if not ids: ids=[tokenizer.unk_token_id]
    if len(ids)>CHUNK_LEN:
        if view_idx==0:
            ids=ids[:CHUNK_LEN]
        elif view_idx==2:
            ids=ids[-CHUNK_LEN:]
        else:
            st=(len(ids)-CHUNK_LEN)//2
            ids=ids[st:st+CHUNK_LEN]
    return ids

def cache_split(name,records):
    ids_path=CACHE_DIR/f"{name}_ids.npy"
    len_path=CACHE_DIR/f"{name}_lens.npy"
    meta_path=CACHE_DIR/f"{name}_meta.json"

    dataset_key="dev" if name.startswith("dev") else name
    fingerprint={
        "dataset_sha256":DATA_SHA256[dataset_key],
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "n":len(records),
        "n_chunks":N_CHUNKS,
        "chunk_len":CHUNK_LEN,
        "char_window":CHAR_WINDOW,
        "views":VIEW_NAMES,
        "ids_hash":hashlib.sha256("|".join(str(r["id"]) for r in records).encode()).hexdigest(),
    }

    if ids_path.exists() and len_path.exists() and meta_path.exists():
        if json.load(open(meta_path))==fingerprint:
            print("✅ reuse token cache:",name)
            return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

    arr=np.full((len(records),N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=np.uint16)
    lens=np.zeros((len(records),N_CHUNKS),dtype=np.uint16)

    t0=time.time()
    for i,r in enumerate(records):
        for c,txt in enumerate(select_views(r["text"])):
            z=encode_view(txt,c)
            L=min(len(z),CHUNK_LEN)
            arr[i,c,:L]=np.asarray(z[:L],dtype=np.uint16)
            lens[i,c]=L
        if (i+1)%2500==0 or i+1==len(records):
            print(f"{name}: {i+1}/{len(records)} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    np.save(ids_path,arr); np.save(len_path,lens)
    json.dump(fingerprint,open(meta_path,"w"),indent=2)
    del arr,lens; gc.collect()
    return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

train_tok,train_lens=cache_split("train",train_records)
official_tok,official_lens=cache_split("dev",dev_raw)
test_tok,test_lens=cache_split("test",test_records)

print("cache shapes:",train_tok.shape,official_tok.shape,test_tok.shape)
assert train_tok.shape==(len(train_records),N_CHUNKS,CHUNK_LEN)


train: 2500/42750 elapsed=0.1m
train: 5000/42750 elapsed=0.2m
train: 7500/42750 elapsed=0.4m
train: 10000/42750 elapsed=0.5m
train: 12500/42750 elapsed=0.6m
train: 15000/42750 elapsed=0.7m
train: 17500/42750 elapsed=0.8m
train: 20000/42750 elapsed=0.9m
train: 22500/42750 elapsed=1.1m
train: 25000/42750 elapsed=1.2m
train: 27500/42750 elapsed=1.3m
train: 30000/42750 elapsed=1.4m
train: 32500/42750 elapsed=1.5m
train: 35000/42750 elapsed=1.6m
train: 37500/42750 elapsed=1.7m
train: 40000/42750 elapsed=1.9m
train: 42500/42750 elapsed=2.0m
train: 42750/42750 elapsed=2.0m
dev: 2500/10181 elapsed=0.1m
dev: 5000/10181 elapsed=0.2m
dev: 7500/10181 elapsed=0.4m
dev: 10000/10181 elapsed=0.5m
dev: 10181/10181 elapsed=0.5m
test: 2500/13019 elapsed=0.1m
test: 5000/13019 elapsed=0.3m
test: 7500/13019 elapsed=0.4m
test: 10000/13019 elapsed=0.5m
test: 12500/13019 elapsed=0.6m
test: 13019/13019 elapsed=0.7m
cache shapes: (42750, 3, 320) (10181, 3, 320) (13019, 3, 320)


In [7]:
# CELL 7 — exact Phase-1 Model4 architecture
# CELL 7 — exact Phase-1 Model4 classes (copied from supplied Phase-1 notebook)
# Cell 9 — ~99.31M Efficient Modern Decoder: GQA + RoPE + RMSNorm + SwiGLU   (v2)
# Same parameters / same state_dict keys / same maths as v1 (v1 checkpoints load unchanged). What changed:
#   * RoPE cos/sin are gathered ONCE per forward (v1: once per layer = 34 gathers + 17 GPU->CPU syncs per step)
#   * decode steps use the GQA-native path: the 5 query heads of a KV group become 5 query rows, so the cached
#     K/V are never expanded 5x (v1 copied the whole expanded cache every layer at every generated token)
#   * multi-token chunks that follow a cache are now causal (v1 silently made them bidirectional; unused so far)
#   * build_model() so training always starts from fresh weights; embeddings excluded from weight decay in Cell 11

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32.to(dtype) * self.weight)


class RotaryTable(nn.Module):
    """cos/sin table (non-persistent buffers -> nothing is added to the state_dict)."""
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        # (B,T) -> two (B,1,T,D/2) tensors, shared by every layer
        return (self.cos_cached[position_ids].unsqueeze(1),
                self.sin_cached[position_ids].unsqueeze(1))


def apply_rope(x, cos, sin):
    # x: (B,H,T,D) with interleaved (even, odd) rotation pairs -- identical convention to v1
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True     # flip to False to fall back to the (slower) expand-K/V path; used by the self-test

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None          # compact cache: only KV heads are stored
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        # ---------------- single new token (cached decoding): GQA-native, no K/V expansion
        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)     # head h = kv*rep + r  ->  row r of group kv
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False)
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        # ---------------- general path (training / prefill / chunks)
        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True                             # fast path used by training
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:                        # left-padding queries keep one safe key (themselves)
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn, attn_mask=attn_mask, dropout_p=drop, is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache, key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(self, input_ids, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"context {past_len + T} exceeds max_positions={self.max_positions}")

        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(position_ids)               # once for all layers (callers keep ids < max_positions)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, cos, sin, past_kv=past, use_cache=use_cache, key_valid=key_valid)
            if use_cache:
                new_cache.append(kv)
        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)   # tied token/output embedding




In [8]:
# CELL 8 — load Model4 + inject true LoRMA+ multiplicative adapters
def build_backbone():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer),d_model=D_MODEL,n_layers=N_LAYERS,
        n_heads=N_HEADS,n_kv_heads=N_KV_HEADS,ffn_dim=FFN_DIM,
        dropout=DROPOUT,max_positions=MAX_POSITIONS,
        rope_base=ROPE_BASE,rms_eps=RMS_EPS
    )

backbone=build_backbone()
obj=torch.load(MODEL4_CKPT,map_location="cpu",weights_only=False)
assert isinstance(obj,dict) and "model" in obj
missing,unexpected=backbone.load_state_dict(obj["model"],strict=False)
assert not missing and not unexpected,(missing,unexpected)
print("✅ Model4 load PASS | params=%.3fM"%(sum(p.numel() for p in backbone.parameters())/1e6))
print("checkpoint global_step:",obj.get("global_step"),"val_loss:",obj.get("val_loss"))

for p in backbone.parameters():
    p.requires_grad=False

class LoRMAPlusLinear(nn.Module):
    """
    Multiplicative LoRMA+:
        y0 = W0 x
        y  = y0 + (alpha/r) B(A(y0))
    = (I + alpha/r BA) W0 x

    W0 is frozen. B starts at zero, so initialization exactly equals W0.
    """
    def __init__(self,base_linear,rank=8,alpha=16.0,init_std=0.02):
        super().__init__()
        assert isinstance(base_linear,nn.Linear)
        self.base=base_linear
        for p in self.base.parameters(): p.requires_grad=False

        self.rank=int(rank)
        self.alpha=float(alpha)
        self.scaling=self.alpha/self.rank
        d_out=int(base_linear.out_features)

        self.A=nn.Linear(d_out,self.rank,bias=False)
        self.B=nn.Linear(self.rank,d_out,bias=False)
        nn.init.normal_(self.A.weight,mean=0.0,std=init_std)
        nn.init.zeros_(self.B.weight)

    def forward(self,x):
        y0=self.base(x)
        return y0+self.scaling*self.B(self.A(y0))

def inject_lorma_plus(backbone):
    names=[]
    for bi in ADAPTED_BLOCKS:
        blk=backbone.blocks[bi]
        groups=[("attn",blk.attn,LORMA_TARGETS)]
        if LORMA_ADAPT_FFN: groups.append(("ff",blk.ff,LORMA_FF_TARGETS))   # v2
        for gname,mod,targets in groups:
            for name in targets:
                old=getattr(mod,name)
                assert isinstance(old,nn.Linear),(bi,gname,name,type(old))
                setattr(mod,name,LoRMAPlusLinear(old,LORMA_RANK,LORMA_ALPHA,LORMA_INIT_STD))
                names.append(f"blocks.{bi}.{gname}.{name}")
    return names

# Identity-at-initialization test on CPU.
backbone.eval()
with torch.inference_mode():
    probe=torch.tensor([[4,5,6,7,8,9]],dtype=torch.long)
    h0,_=backbone.forward_hidden(probe)

injected_modules=inject_lorma_plus(backbone)

with torch.inference_mode():
    h1,_=backbone.forward_hidden(probe)

init_diff=(h0-h1).abs().max().item()
print("Injected modules:",len(injected_modules))
print("max init hidden diff:",init_diff)
assert init_diff<1e-6

bad=[
    n for n,p in backbone.named_parameters()
    if p.requires_grad and not (".A.weight" in n or ".B.weight" in n)
]
assert not bad,bad

print("LoRMA backbone trainable params:",
      sum(p.numel() for p in backbone.parameters() if p.requires_grad))


✅ Model4 load PASS | params=99.309M
checkpoint global_step: 16266 val_loss: 1.3664915735571412
Injected modules: 56
max init hidden diff: 0.0
LoRMA backbone trainable params: 1605632


In [9]:
# CELL 9 — lightweight 3-view hierarchical classifier
class StatuteLoRMAClassifier(nn.Module):
    def __init__(self,backbone,d_model=640,proj_dim=320,n_labels=100,dropout=0.10):
        super().__init__()
        self.backbone=backbone
        self.proj=nn.Sequential(
            nn.Linear(2*d_model,proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )
        self.chunk_pos=nn.Parameter(torch.zeros(N_CHUNKS,proj_dim))
        nn.init.normal_(self.chunk_pos,std=0.02)
        self.attn=nn.Sequential(
            nn.Linear(proj_dim,96),nn.Tanh(),nn.Linear(96,1)
        )
        self.head=nn.Sequential(
            nn.Linear(2*proj_dim,proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim,n_labels),
        )

    def encode_flat_chunks(self,ids,valid):
        B,T=ids.shape
        position_ids=torch.arange(T,device=ids.device)[None,:].expand(B,T)
        cos,sin=self.backbone.rope(position_ids)
        kv=None if FAST_CAUSAL_NO_MASK else valid

        # Bottom nine blocks are entirely frozen -> no backward graph.
        with torch.no_grad():
            x=self.backbone.drop(self.backbone.token_emb(ids))
            for block in self.backbone.blocks[:ADAPT_START_BLOCK]:
                x,_=block(x,cos,sin,past_kv=None,use_cache=False,key_valid=kv)
        x=x.detach()

        # Top eight blocks contain trainable LoRMA A/B matrices.
        for block in self.backbone.blocks[ADAPT_START_BLOCK:]:
            x,_=block(x,cos,sin,past_kv=None,use_cache=False,key_valid=kv)

        x=self.backbone.final_norm(x)

        mask=valid.unsqueeze(-1).to(x.dtype)
        mean=(x*mask).sum(1)/mask.sum(1).clamp_min(1.0)
        last_idx=valid.long().sum(1).clamp_min(1)-1
        last=x[torch.arange(B,device=x.device),last_idx]
        return self.proj(torch.cat([last,mean],-1))

    def forward(self,input_ids,valid):
        B,C,T=input_ids.shape
        z=self.encode_flat_chunks(
            input_ids.reshape(B*C,T),valid.reshape(B*C,T)
        ).reshape(B,C,-1)
        z=z+self.chunk_pos[None,:,:]
        scores=self.attn(z).squeeze(-1)
        weights=torch.softmax(scores,-1)
        attn_pool=(z*weights.unsqueeze(-1)).sum(1)
        max_pool=z.max(1).values
        logits=self.head(torch.cat([attn_pool,max_pool],-1))
        return logits,weights

model=StatuteLoRMAClassifier(backbone).to(DEVICE)

total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_trainable=sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
print(f"total={total/1e6:.3f}M")
print(f"trainable={trainable/1e6:.3f}M ({100*trainable/total:.3f}%)")
print(f"LoRMA backbone trainable={backbone_trainable/1e6:.3f}M")
assert 0<trainable<total and backbone_trainable>0


total=101.595M
trainable=2.285M (2.249%)
LoRMA backbone trainable=1.606M


In [10]:
# CELL 10 — datasets/loaders
class LSIDataset(Dataset):
    def __init__(self,records,tok,lens,labeled=True):
        self.records=records; self.tok=tok; self.lens=lens; self.labeled=labeled
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        ids=torch.from_numpy(np.array(self.tok[i],dtype=np.int64,copy=True))
        lens=torch.from_numpy(np.array(self.lens[i],dtype=np.int64,copy=True))
        valid=torch.arange(CHUNK_LEN)[None,:]<lens[:,None]
        out={"input_ids":ids,"valid":valid,"id":str(self.records[i]["id"])}
        if self.labeled:
            y=torch.zeros(N_LABELS,dtype=torch.float32)
            y[self.records[i]["labels"]]=1.0
            out["labels"]=y
        return out

def make_loader(records,tok,lens,batch,shuffle,labeled=True,seed=SEED):
    # v2: persistent workers only for the training loader (dev/test loaders are rebuilt cheaply & survive interrupts)
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(
        LSIDataset(records,tok,lens,labeled),
        batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0 and shuffle),
        generator=g
    )

train_loader=make_loader(train_records,train_tok,train_lens,MICRO_BATCH_SIZE,True,True)
official_loader=make_loader(dev_raw,official_tok,official_lens,DEV_BATCH_SIZE,False,True)
test_loader=make_loader(test_records,test_tok,test_lens,DEV_BATCH_SIZE,False,False)

print("loader batches:",len(train_loader),len(official_loader),len(test_loader))
print("train tokens/microbatch:",MICRO_BATCH_SIZE*N_CHUNKS*CHUNK_LEN)


loader batches: 10688 637 814
train tokens/microbatch: 3840


In [11]:
# CELL 11 — exact evaluator logic + threshold tuning
def multilabel_metrics(y_true,y_pred):
    return {
        "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(y_true,y_pred,average="micro",zero_division=0)),
        "subset_accuracy":float(accuracy_score(y_true,y_pred)),
    }

def apply_thresholds(prob,thresholds,ensure_nonempty=True):
    thr=np.asarray(thresholds,dtype=np.float32)
    pred=prob>=thr[None,:]
    if ensure_nonempty:
        empty=np.flatnonzero(~pred.any(1))
        if len(empty):
            best=np.argmax(prob[empty]/np.maximum(thr[None,:],1e-6),axis=1)
            pred[empty,best]=True
    return pred

def macro_f1_fast(y,pred):
    yb=y.astype(bool); pb=pred.astype(bool)
    tp=(yb&pb).sum(0).astype(np.float64)
    den=yb.sum(0)+pb.sum(0)
    return float(np.where(den>0,2*tp/np.maximum(den,1),0.0).mean())

def f1_curves(y,prob,grid=THR_GRID):
    yb=y.astype(bool); P=yb.sum(0).astype(np.float64)
    out=np.zeros((len(grid),y.shape[1]),dtype=np.float64)
    for gi,t in enumerate(grid):
        pr=prob>=t
        tp=(pr&yb).sum(0); den=pr.sum(0)+P
        out[gi]=np.where(den>0,2*tp/np.maximum(den,1),0.0)
    return out

def smooth_curves(c):
    p=np.pad(c,((1,1),(0,0)),mode="edge")
    return (p[:-2]+p[1:-1]+p[2:])/3.0

def fit_threshold_strategies(y,prob):
    c=f1_curves(y,prob)
    g=float(THR_GRID[int(np.argmax(c.mean(1)))])
    per=THR_GRID[np.argmax(smooth_curves(c),axis=0)].astype(np.float32)
    P=y.sum(0).astype(np.float64)
    out={"global":np.full(y.shape[1],g,dtype=np.float32),"per_label":per}
    for tau in SHRINK_TAUS:
        w=P/(P+tau)
        out[f"shrunk_tau_{int(tau)}"]=(w*per+(1-w)*g).astype(np.float32)
    return out,g,per

def optimize_thresholds(y,prob):
    full,g,per=fit_threshold_strategies(y,prob)
    n=len(y); cv={k:[] for k in full}
    for rep_i in range(CV_REPEATS):
        perm=np.random.RandomState(100+rep_i).permutation(n)
        a,b=perm[:n//2],perm[n//2:]
        for tr,te in ((a,b),(b,a)):
            strat,_,_=fit_threshold_strategies(y[tr],prob[tr])
            for k,thr in strat.items():
                cv[k].append(
                    macro_f1_fast(y[te],apply_thresholds(prob[te],thr,True))
                )
    cv={k:float(np.mean(v)) for k,v in cv.items()}
    name=max(cv,key=cv.get); thr=full[name]
    return {
        "name":name,"thresholds":thr,"cv_macro_f1":cv[name],
        "insample_macro_f1":macro_f1_fast(y,apply_thresholds(prob,thr,True)),
        "cv_all":cv
    },g,per

@torch.inference_mode()
def predict_loader(model,loader):
    model.eval()
    ids=[]; probs=[]; ys=[]; atts=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast(
            "cuda",dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP)
        ):
            lg,a=model(x,v)
        assert torch.isfinite(lg).all()
        ids.extend(list(b["id"]))
        probs.append(torch.sigmoid(lg.float()).cpu().numpy())
        atts.append(a.float().cpu().numpy())
        if "labels" in b: ys.append(b["labels"].numpy().astype(np.uint8))
    return ids,np.concatenate(probs),(np.concatenate(ys) if ys else None),np.concatenate(atts)

def make_pred_records(ids,prob,thresholds,ensure_nonempty=True):
    binary=apply_thresholds(prob,thresholds,ensure_nonempty)
    return [{"id":str(i),"pred":[int(x) for x in np.flatnonzero(binary[k])]} for k,i in enumerate(ids)]

def exact_phase5_score(truth_records,pred_records):
    t={}; max_label=-1
    for r in truth_records:
        assert r["id"] not in t
        seen=set()
        for label in r["gold"]:
            assert isinstance(label,int) and not isinstance(label,bool) and label>=0 and label not in seen
            seen.add(label); max_label=max(max_label,label)
        t[r["id"]]=r["gold"]

    nlab=max_label+1
    p={}
    for r in pred_records:
        assert r["id"] not in p
        seen=set()
        for x in r["pred"]:
            assert isinstance(x,int) and not isinstance(x,bool) and 0<=x<nlab and x not in seen
            seen.add(x)
        p[r["id"]]=r["pred"]

    assert set(t)==set(p)
    yt=np.zeros((len(t),nlab),dtype=np.uint8)
    yp=np.zeros((len(t),nlab),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    return multilabel_metrics(yt,yp)

print("✅ metric/threshold utilities ready")


✅ metric/threshold utilities ready


In [12]:
# CELL 12 — LoRMA+ training: 2 epochs, ONE DEV pass/epoch
pos_weight_t=torch.tensor(POS_WEIGHT,dtype=torch.float32,device=DEVICE)

lorma_params=[]; head_params=[]
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    if n.startswith("backbone.") and (".A.weight" in n or ".B.weight" in n):
        lorma_params.append(p)
    else:
        head_params.append(p)

assert lorma_params and head_params
print("LoRMA params:",sum(p.numel() for p in lorma_params))
print("Head params:",sum(p.numel() for p in head_params))

optimizer=torch.optim.AdamW([
    {"params":lorma_params,"lr":LORMA_LR,"weight_decay":0.0},
    {"params":head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
],betas=(0.9,0.999),eps=1e-8)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=updates_per_epoch*EPOCHS
warmup=max(1,int(WARMUP_RATIO*total_updates))

def lr_lambda(step):
    if step<warmup: return (step+1)/warmup
    q=(step-warmup)/max(1,total_updates-warmup)
    return max(0.08,0.5*(1+math.cos(math.pi*min(1.0,q))))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda" and USE_AMP))

BEST=ROOT/"best_exp2_lormaplus.pt"
HISTORY=ROOT/"history.json"

class EMA:
    def __init__(self,model,decay):
        self.decay=decay
        self.params={n:p for n,p in model.named_parameters() if p.requires_grad}
        self.shadow={n:p.detach().clone() for n,p in self.params.items()}
        self.n=0
    @torch.no_grad()
    def update(self):
        self.n+=1
        d=min(self.decay,(1.0+self.n)/(10.0+self.n))
        for n,p in self.params.items():
            self.shadow[n].mul_(d).add_(p.detach(),alpha=1.0-d)
    @contextlib.contextmanager
    def applied(self):
        backup={n:p.detach().clone() for n,p in self.params.items()}
        with torch.no_grad():
            for n,p in self.params.items(): p.copy_(self.shadow[n])
        try: yield
        finally:
            with torch.no_grad():
                for n,p in self.params.items(): p.copy_(backup[n])

def get_trainable_state(model):
    return {
        n:p.detach().cpu().clone()
        for n,p in model.named_parameters() if p.requires_grad
    }

def load_trainable_state(model,state):
    named=dict(model.named_parameters())
    with torch.no_grad():
        for n,v in state.items():
            assert n in named,n
            named[n].copy_(v.to(named[n].device,dtype=named[n].dtype))

ema=EMA(model,EMA_DECAY)
best_score=-1.0; history=[]; global_update=0
optimizer.zero_grad(set_to_none=True)

def evaluate_epoch(epoch,tag="ema"):
    global best_score

    # One forward pass over official DEV; clean DEV is a slice.
    oids,oprob,oy,oatt=predict_loader(model,official_loader)
    cprob=oprob[CLEAN_IDX]; cy=oy[CLEAN_IDX]
    cids=[oids[i] for i in CLEAN_IDX]; catt=oatt[CLEAN_IDX]

    tune,g,per=optimize_thresholds(cy,cprob)
    clean_pred=make_pred_records(cids,cprob,tune["thresholds"],True)
    official_pred=make_pred_records(oids,oprob,tune["thresholds"],True)
    exact_clean=exact_phase5_score(clean_truth,clean_pred)
    exact_official=exact_phase5_score(official_truth,official_pred)

    row={
        "epoch":epoch,"weights":tag,"strategy":tune["name"],
        "global_threshold":g,"cv_macro_f1":tune["cv_macro_f1"],
        "insample_macro_f1":tune["insample_macro_f1"],
        "cv_all":tune["cv_all"],"clean":exact_clean,"official":exact_official,
        "mean_pred_labels_clean":float(np.mean([len(r["pred"]) for r in clean_pred])),
        "mean_gold_labels_clean":float(cy.sum(1).mean()),
        "attention_mean_clean":catt.mean(0).tolist(),
    }
    history.append(row); json.dump(history,open(HISTORY,"w"),indent=2)

    print(
        f"[{tag}] epoch={epoch} strat={tune['name']} CV-macroF1={tune['cv_macro_f1']:.4f} "
        f"clean={exact_clean['macro_f1']:.4f}/{exact_clean['micro_f1']:.4f}/{exact_clean['subset_accuracy']:.4f} "
        f"official-macro={exact_official['macro_f1']:.4f} "
        f"pred/gold={row['mean_pred_labels_clean']:.2f}/{row['mean_gold_labels_clean']:.2f}",
        flush=True
    )

    if tune["cv_macro_f1"]>best_score:
        best_score=tune["cv_macro_f1"]
        atomic_torch_save({
            "trainable_state":get_trainable_state(model),
            "thresholds":np.asarray(tune["thresholds"],dtype=np.float32),
            "threshold_strategy":tune["name"],
            "global_threshold":g,
            "per_label_thresholds":np.asarray(per,dtype=np.float32),
            "cv_macro_f1":tune["cv_macro_f1"],
            "clean_metrics":exact_clean,
            "official_metrics":exact_official,
            "epoch":epoch,"weights_tag":tag,
            "model4_sha256":sha256_file(MODEL4_CKPT),
            "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
            "dataset_sha256":DATA_SHA256,
            "config":{
                "method":"LoRMA+","rank":LORMA_RANK,"alpha":LORMA_ALPHA,
                "adapt_start_block":ADAPT_START_BLOCK,
                "targets":list(LORMA_TARGETS),"ffn_adapted":LORMA_ADAPT_FFN,"ff_targets":list(LORMA_FF_TARGETS),
                "n_chunks":N_CHUNKS,"chunk_len":CHUNK_LEN,"views":VIEW_NAMES,
                "lorma_lr":LORMA_LR,"head_lr":HEAD_LR,
                "epochs":EPOCHS,"pos_weight":"sqrt(raw) clipped [1,10]"
            }
        },BEST)
        print("✅ NEW BEST:",round(best_score,5))

print("batches/epoch:",len(train_loader),"optimizer updates:",total_updates)
print("OOM fallback: MICRO_BATCH_SIZE=1, GRAD_ACCUM_STEPS=16.")

for epoch in range(1,EPOCHS+1):
    model.train(); t0=time.time(); running=0.0; good=0

    for bi,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            "cuda",dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP)
        ):
            logits,_=model(x,v)
            loss=F.binary_cross_entropy_with_logits(logits,y,pos_weight=pos_weight_t)
            scaled_loss=loss/GRAD_ACCUM_STEPS

        if torch.isfinite(loss):
            scaler.scale(scaled_loss).backward()
            good+=1; running+=float(loss.detach())
        else:
            print("⚠️ non-finite loss batch",bi)

        if bi%GRAD_ACCUM_STEPS==0 or bi==len(train_loader):
            if good:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],GRAD_CLIP
                )
                old_scale=float(scaler.get_scale())
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if float(scaler.get_scale())>=old_scale:
                    scheduler.step(); global_update+=1; ema.update()
            else:
                optimizer.zero_grad(set_to_none=True)
            good=0

        if bi%500==0 or bi==len(train_loader):
            elapsed=(time.time()-t0)/60
            eta=elapsed*(len(train_loader)-bi)/max(1,bi)
            print(
                f"epoch={epoch} batch={bi}/{len(train_loader)} "
                f"loss={running/max(1,bi):.4f} elapsed={elapsed:.1f}m ETA={eta:.1f}m",
                flush=True
            )

    # Major speed difference from Exp 1: only one DEV inference pass per epoch.
    with ema.applied():
        evaluate_epoch(epoch,"ema")

    gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

assert BEST.exists()
print("✅ training complete | best clean-DEV CV Macro-F1:",best_score)


LoRMA params: 1605632
Head params: 679653
batches/epoch: 10688 optimizer updates: 5344
OOM fallback: MICRO_BATCH_SIZE=1, GRAD_ACCUM_STEPS=16.
epoch=1 batch=500/10688 loss=0.5417 elapsed=1.2m ETA=24.7m
epoch=1 batch=1000/10688 loss=0.4755 elapsed=2.5m ETA=23.7m
epoch=1 batch=1500/10688 loss=0.4387 elapsed=3.7m ETA=22.6m
epoch=1 batch=2000/10688 loss=0.4206 elapsed=4.9m ETA=21.4m
epoch=1 batch=2500/10688 loss=0.4062 elapsed=6.2m ETA=20.2m
epoch=1 batch=3000/10688 loss=0.3944 elapsed=7.4m ETA=19.0m
epoch=1 batch=3500/10688 loss=0.3863 elapsed=8.6m ETA=17.8m
epoch=1 batch=4000/10688 loss=0.3788 elapsed=9.9m ETA=16.5m
epoch=1 batch=4500/10688 loss=0.3723 elapsed=11.1m ETA=15.3m
epoch=1 batch=5000/10688 loss=0.3674 elapsed=12.4m ETA=14.1m
epoch=1 batch=5500/10688 loss=0.3636 elapsed=13.6m ETA=12.8m
epoch=1 batch=6000/10688 loss=0.3608 elapsed=14.9m ETA=11.6m
epoch=1 batch=6500/10688 loss=0.3581 elapsed=16.1m ETA=10.4m
epoch=1 batch=7000/10688 loss=0.3554 elapsed=17.3m ETA=9.1m
epoch=1 batch=

In [13]:
# CELL 13 — reload BEST + final heldout diagnostics
best=torch.load(BEST,map_location="cpu",weights_only=False)
load_trainable_state(model,best["trainable_state"])
model.to(DEVICE); model.eval()
thresholds=np.asarray(best["thresholds"],dtype=np.float32)

print("BEST epoch:",best["epoch"])
print("threshold strategy:",best["threshold_strategy"])
print("CV macro-F1:",best["cv_macro_f1"])
print("clean metrics:",best["clean_metrics"])
print("official metrics:",best["official_metrics"])
print("threshold min/median/max:",thresholds.min(),np.median(thresholds),thresholds.max())

dids,dprob,dy,datt=predict_loader(model,official_loader)
cids=[dids[i] for i in CLEAN_IDX]
cprob=dprob[CLEAN_IDX]
re_pred=make_pred_records(cids,cprob,thresholds,True)
re_score=exact_phase5_score(clean_truth,re_pred)
print("re-scored clean DEV:",re_score)
assert abs(re_score["macro_f1"]-best["clean_metrics"]["macro_f1"])<5e-3

print("mean attention [HEAD,EVIDENCE,TAIL]:",datt[CLEAN_IDX].mean(0))
print(
    "mean predicted labels:",
    apply_thresholds(cprob,thresholds,True).sum(1).mean(),
    "gold:",dy[CLEAN_IDX].sum(1).mean()
)

np.savez_compressed(
    ROOT/"dev_probs_exp2.npz",
    id=np.asarray(dids),prob=dprob,y=dy,
    clean_idx=CLEAN_IDX,thresholds=thresholds
)

cy=dy[CLEAN_IDX]
cp=apply_thresholds(cprob,thresholds,True).astype(np.uint8)
per_f1=np.array([f1_score(cy[:,j],cp[:,j],zero_division=0) for j in range(N_LABELS)])
worst=np.argsort(per_f1)[:10]
print("weakest labels:",[(int(j),round(float(per_f1[j]),3),int(train_counts[j])) for j in worst])


BEST epoch: 2
threshold strategy: per_label
CV macro-F1: 0.3348820595890764
clean metrics: {'macro_f1': 0.3505803580523601, 'micro_f1': 0.4400100349131353, 'subset_accuracy': 0.0217628937692537}
official metrics: {'macro_f1': 0.350075118160269, 'micro_f1': 0.4396608631075171, 'subset_accuracy': 0.021707101463510462}
threshold min/median/max: 0.19 0.53 0.8
re-scored clean DEV: {'macro_f1': 0.3505803580523601, 'micro_f1': 0.4400100349131353, 'subset_accuracy': 0.0217628937692537}
mean attention [HEAD,EVIDENCE,TAIL]: [0.34302217 0.29280034 0.3641771 ]
mean predicted labels: 5.59515055152539 gold: 3.9115571897048595
weakest labels: [(8, 0.063, 349), (0, 0.073, 343), (2, 0.103, 367), (65, 0.106, 351), (52, 0.113, 1084), (41, 0.119, 821), (99, 0.137, 674), (25, 0.151, 287), (84, 0.153, 272), (18, 0.154, 393)]


In [14]:
# CELL 14 — exact official Phase-5 evaluator + literal DEV script test
EVAL_SCRIPT = ROOT / "evaluate_phase5.py"
EVAL_SCRIPT.write_text('#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n', encoding="utf-8")
print("✅ wrote exact evaluator:", EVAL_SCRIPT)


✅ wrote exact evaluator: /kaggle/working/phase5_exp2_model4_lorma_fast_v4/evaluate_phase5.py


In [15]:
# CELL 15 — TEST inference + Codabench ZIP + exact parser validation
tids,tprob,_,tatt=predict_loader(model,test_loader)
pred_records=make_pred_records(tids,tprob,thresholds,True)

assert len(pred_records)==len(test_records)
assert len({r["id"] for r in pred_records})==len(test_records)
assert {r["id"] for r in pred_records}=={str(r["id"]) for r in test_records}

for r in pred_records:
    assert len(r["pred"])==len(set(r["pred"]))
    assert all(isinstance(x,int) and not isinstance(x,bool) and 0<=x<N_LABELS for x in r["pred"])

PRED=ROOT/"prediction.json"
json.dump(pred_records,open(PRED,"w",encoding="utf-8"),ensure_ascii=False)

ZIP=ROOT/"submission_exp2_Model4_LORMAplus_FAST.zip"
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED,arcname="prediction.json")

print("TEST mean predicted labels/document:",np.mean([len(r["pred"]) for r in pred_records]))
print("TEST empty:",sum(len(r["pred"])==0 for r in pred_records))
print("TEST attention mean [HEAD,EVIDENCE,TAIL]:",tatt.mean(0))
print("ZIP:",ZIP)

np.savez_compressed(
    ROOT/"test_probs_exp2.npz",
    id=np.asarray(tids),prob=tprob,thresholds=thresholds
)

spec=importlib.util.spec_from_file_location("evaluate_phase5_exact",EVAL_SCRIPT)
eval_exact=importlib.util.module_from_spec(spec)
spec.loader.exec_module(eval_exact)

check_dir=ROOT/"zip_check"
if check_dir.exists(): shutil.rmtree(check_dir)
check_dir.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(ZIP,"r") as z:
    names=z.namelist()
    assert names==["prediction.json"],names
    z.extract("prediction.json",check_dir)

zip_pred=eval_exact.read_pred(str(check_dir/"prediction.json"),N_LABELS)
expected_ids={str(r["id"]) for r in test_records}
assert set(zip_pred)==expected_ids
print("✅ FINAL ZIP PASS — ready for Codabench")


TEST mean predicted labels/document: 5.524387433750672
TEST empty: 0
TEST attention mean [HEAD,EVIDENCE,TAIL]: [0.34296146 0.29321656 0.36382055]
ZIP: /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_FAST.zip
✅ FINAL ZIP PASS — ready for Codabench


## Fresh TRAIN+DEV refit + heldout/refit ensemble

This stage is deliberately **fresh**:

- reload original Phase-1 Model4
- inject the same LoRMA+ configuration
- initialize a fresh Phase-5 classifier
- train on **TRAIN + DEV**
- use the best heldout-selected epoch count
- perform **no DEV evaluation during refit**
- infer TEST once
- create a refit-only ZIP
- average heldout/refit TEST probabilities 50/50 and create an ensemble ZIP

Thresholds come from the honest heldout run. We do **not** tune refit thresholds on DEV after DEV has entered refit training.


In [16]:
# CELL 16 — fresh TRAIN+DEV LoRMA+ refit (no validation passes)
REFIT_EPOCHS = max(1, int(best["epoch"]))
REFIT_ENSEMBLE_WEIGHT = 0.50   # probability weight on refit model
REFIT_SEED = SEED + 777

print("Fresh refit epochs:", REFIT_EPOCHS)
print("TRAIN+DEV rows:", len(train_records) + len(dev_raw))
print("Heldout/refit ensemble weights:",
      1.0 - REFIT_ENSEMBLE_WEIGHT, "/", REFIT_ENSEMBLE_WEIGHT)

# ---- fresh model from original Phase-1 checkpoint ----
torch.manual_seed(REFIT_SEED)
np.random.seed(REFIT_SEED)
random.seed(REFIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(REFIT_SEED)

refit_backbone = build_backbone()
phase1_refit = torch.load(MODEL4_CKPT, map_location="cpu", weights_only=False)
missing, unexpected = refit_backbone.load_state_dict(phase1_refit["model"], strict=False)
assert not missing and not unexpected, (missing, unexpected)

for p in refit_backbone.parameters():
    p.requires_grad = False

refit_injected = inject_lorma_plus(refit_backbone)
assert len(refit_injected) == len(injected_modules)

refit_model = StatuteLoRMAClassifier(refit_backbone).to(DEVICE)

# Verify only LoRMA + task head are trainable.
refit_lorma_params = []
refit_head_params = []
for n, p in refit_model.named_parameters():
    if not p.requires_grad:
        continue
    if n.startswith("backbone.") and (".A.weight" in n or ".B.weight" in n):
        refit_lorma_params.append(p)
    else:
        refit_head_params.append(p)

assert refit_lorma_params and refit_head_params
print("refit LoRMA trainable:", sum(p.numel() for p in refit_lorma_params))
print("refit head trainable :", sum(p.numel() for p in refit_head_params))

# ---- TRAIN+DEV data without rebuilding token caches ----
refit_train_ds = torch.utils.data.ConcatDataset([
    LSIDataset(train_records, train_tok, train_lens, labeled=True),
    LSIDataset(dev_raw, official_tok, official_lens, labeled=True),
])

refit_gen = torch.Generator()
refit_gen.manual_seed(REFIT_SEED)
refit_loader = DataLoader(
    refit_train_ds,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    generator=refit_gen,
)

# Recompute imbalance weights on TRAIN+DEV.
full_counts = np.zeros(N_LABELS, dtype=np.int64)
for r in train_records:
    full_counts[r["labels"]] += 1
for r in dev_raw:
    full_counts[r["labels"]] += 1

full_n = len(train_records) + len(dev_raw)
full_raw_pw = (full_n - full_counts) / full_counts
REFIT_POS_WEIGHT = np.clip(np.sqrt(full_raw_pw), 1.0, 10.0).astype(np.float32)
refit_pos_weight_t = torch.tensor(
    REFIT_POS_WEIGHT, dtype=torch.float32, device=DEVICE
)

refit_optimizer = torch.optim.AdamW(
    [
        {"params": refit_lorma_params, "lr": LORMA_LR, "weight_decay": 0.0},
        {"params": refit_head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
    ],
    betas=(0.9, 0.999),
    eps=1e-8,
)

refit_updates_per_epoch = math.ceil(len(refit_loader) / GRAD_ACCUM_STEPS)
refit_total_updates = refit_updates_per_epoch * REFIT_EPOCHS
refit_warmup = max(1, int(WARMUP_RATIO * refit_total_updates))

def refit_lr_lambda(step):
    if step < refit_warmup:
        return (step + 1) / refit_warmup
    q = (step - refit_warmup) / max(1, refit_total_updates - refit_warmup)
    return max(0.08, 0.5 * (1 + math.cos(math.pi * min(1.0, q))))

refit_scheduler = torch.optim.lr_scheduler.LambdaLR(
    refit_optimizer, refit_lr_lambda
)
refit_scaler = torch.amp.GradScaler(
    "cuda", enabled=(DEVICE.type == "cuda" and USE_AMP)
)
refit_ema = EMA(refit_model, EMA_DECAY)

REFIT_STATE = ROOT / "refit_trainable_state.pt"

refit_optimizer.zero_grad(set_to_none=True)
refit_global_update = 0

for epoch in range(1, REFIT_EPOCHS + 1):
    refit_model.train()
    t0 = time.time()
    running = 0.0
    good = 0

    for bi, b in enumerate(refit_loader, 1):
        x = b["input_ids"].to(DEVICE, non_blocking=True)
        v = b["valid"].to(DEVICE, non_blocking=True)
        y = b["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda" and USE_AMP),
        ):
            logits, _ = refit_model(x, v)
            loss = F.binary_cross_entropy_with_logits(
                logits, y, pos_weight=refit_pos_weight_t
            )
            scaled_loss = loss / GRAD_ACCUM_STEPS

        if torch.isfinite(loss):
            refit_scaler.scale(scaled_loss).backward()
            running += float(loss.detach())
            good += 1
        else:
            print("⚠️ refit non-finite loss at batch", bi)

        if bi % GRAD_ACCUM_STEPS == 0 or bi == len(refit_loader):
            if good:
                refit_scaler.unscale_(refit_optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in refit_model.parameters() if p.requires_grad],
                    GRAD_CLIP,
                )
                old_scale = float(refit_scaler.get_scale())
                refit_scaler.step(refit_optimizer)
                refit_scaler.update()
                refit_optimizer.zero_grad(set_to_none=True)

                if float(refit_scaler.get_scale()) >= old_scale:
                    refit_scheduler.step()
                    refit_global_update += 1
                    refit_ema.update()
            else:
                refit_optimizer.zero_grad(set_to_none=True)
            good = 0

        if bi % 500 == 0 or bi == len(refit_loader):
            elapsed = (time.time() - t0) / 60
            eta = elapsed * (len(refit_loader) - bi) / max(1, bi)
            print(
                f"REFIT epoch={epoch}/{REFIT_EPOCHS} "
                f"batch={bi}/{len(refit_loader)} "
                f"loss={running/max(1,bi):.4f} "
                f"elapsed={elapsed:.1f}m ETA={eta:.1f}m",
                flush=True,
            )

# Save EMA refit trainable state and infer TEST only once.
with refit_ema.applied():
    atomic_torch_save(
        {
            "trainable_state": get_trainable_state(refit_model),
            "epochs": REFIT_EPOCHS,
            "global_updates": refit_global_update,
            "model4_sha256": sha256_file(MODEL4_CKPT),
            "tokenizer_sha256": sha256_file(TOKENIZER_FILE),
            "thresholds_from_heldout": thresholds,
        },
        REFIT_STATE,
    )
    rids, rprob, _, ratt = predict_loader(refit_model, test_loader)

assert list(map(str, rids)) == list(map(str, tids))

np.savez_compressed(
    ROOT / "test_probs_refit_exp2.npz",
    id=np.asarray(rids),
    prob=rprob,
    thresholds=thresholds,
)

print("✅ fresh TRAIN+DEV refit complete")
print("REFIT TEST attention mean [HEAD,EVIDENCE,TAIL]:", ratt.mean(0))


Fresh refit epochs: 2
TRAIN+DEV rows: 52931
Heldout/refit ensemble weights: 0.5 / 0.5
refit LoRMA trainable: 1605632
refit head trainable : 679653
REFIT epoch=1/2 batch=500/13233 loss=0.5521 elapsed=1.2m ETA=31.7m
REFIT epoch=1/2 batch=1000/13233 loss=0.4849 elapsed=2.5m ETA=30.4m
REFIT epoch=1/2 batch=1500/13233 loss=0.4507 elapsed=3.7m ETA=29.1m
REFIT epoch=1/2 batch=2000/13233 loss=0.4290 elapsed=5.0m ETA=27.9m
REFIT epoch=1/2 batch=2500/13233 loss=0.4125 elapsed=6.2m ETA=26.7m
REFIT epoch=1/2 batch=3000/13233 loss=0.4035 elapsed=7.5m ETA=25.4m
REFIT epoch=1/2 batch=3500/13233 loss=0.3935 elapsed=8.7m ETA=24.2m
REFIT epoch=1/2 batch=4000/13233 loss=0.3863 elapsed=9.9m ETA=22.9m
REFIT epoch=1/2 batch=4500/13233 loss=0.3815 elapsed=11.2m ETA=21.7m
REFIT epoch=1/2 batch=5000/13233 loss=0.3769 elapsed=12.4m ETA=20.5m
REFIT epoch=1/2 batch=5500/13233 loss=0.3727 elapsed=13.7m ETA=19.2m
REFIT epoch=1/2 batch=6000/13233 loss=0.3690 elapsed=14.9m ETA=18.0m
REFIT epoch=1/2 batch=6500/13233 l

In [17]:
# CELL 17 — create REFIT and HELDOUT+REFIT ENSEMBLE submission ZIPs
def validate_submission_zip(zip_path, expected_ids):
    tmp = ROOT / ("check_" + zip_path.stem)
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        assert names == ["prediction.json"], names
        z.extract("prediction.json", tmp)

    parsed = eval_exact.read_pred(str(tmp / "prediction.json"), N_LABELS)
    assert set(parsed) == expected_ids
    return parsed

expected_ids = {str(r["id"]) for r in test_records}

# ---- 1) Refit-only ----
refit_records = make_pred_records(
    rids, rprob, thresholds, ensure_nonempty=True
)
REFIT_PRED = ROOT / "prediction_refit.json"
json.dump(
    refit_records,
    open(REFIT_PRED, "w", encoding="utf-8"),
    ensure_ascii=False,
)
REFIT_ZIP = ROOT / "submission_exp2_Model4_LORMAplus_REFIT.zip"
with zipfile.ZipFile(REFIT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(REFIT_PRED, arcname="prediction.json")

validate_submission_zip(REFIT_ZIP, expected_ids)
print(
    "✅ REFIT ZIP PASS:",
    REFIT_ZIP,
    "| mean labels/doc:",
    np.mean([len(r["pred"]) for r in refit_records]),
)

# ---- 2) Heldout + refit probability ensemble ----
ensemble_prob = (
    (1.0 - REFIT_ENSEMBLE_WEIGHT) * tprob
    + REFIT_ENSEMBLE_WEIGHT * rprob
)

ensemble_records = make_pred_records(
    tids, ensemble_prob, thresholds, ensure_nonempty=True
)
ENSEMBLE_PRED = ROOT / "prediction_heldout_refit_ensemble.json"
json.dump(
    ensemble_records,
    open(ENSEMBLE_PRED, "w", encoding="utf-8"),
    ensure_ascii=False,
)
ENSEMBLE_ZIP = ROOT / "submission_exp2_Model4_LORMAplus_HELDOUT_REFIT_ENSEMBLE.zip"
with zipfile.ZipFile(ENSEMBLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ENSEMBLE_PRED, arcname="prediction.json")

validate_submission_zip(ENSEMBLE_ZIP, expected_ids)

np.savez_compressed(
    ROOT / "test_probs_heldout_refit_ensemble.npz",
    id=np.asarray(tids),
    prob=ensemble_prob,
    thresholds=thresholds,
    refit_weight=REFIT_ENSEMBLE_WEIGHT,
)

print(
    "✅ HELDOUT+REFIT ENSEMBLE ZIP PASS:",
    ENSEMBLE_ZIP,
    "| mean labels/doc:",
    np.mean([len(r["pred"]) for r in ensemble_records]),
)

print("\nCORE SUBMISSION CANDIDATES:")
print("  1. HELDOUT :", ZIP)
print("  2. REFIT   :", REFIT_ZIP)
print("  3. ENSEMBLE:", ENSEMBLE_ZIP)


✅ REFIT ZIP PASS: /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_REFIT.zip | mean labels/doc: 5.706352254397419
✅ HELDOUT+REFIT ENSEMBLE ZIP PASS: /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_HELDOUT_REFIT_ENSEMBLE.zip | mean labels/doc: 5.398571318841693

CORE SUBMISSION CANDIDATES:
  1. HELDOUT : /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_FAST.zip
  2. REFIT   : /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_REFIT.zip
  3. ENSEMBLE: /kaggle/working/phase5_exp2_model4_lorma_fast_v4/submission_exp2_Model4_LORMAplus_HELDOUT_REFIT_ENSEMBLE.zip


In [18]:
# CELL 18 — manifest for report/viva
manifest={
    "phase":5,
    "experiment":2,
    "version":NOTEBOOK_VERSION,
    "task":"Legal Statute Identification",
    "method":"LoRMA+ (Low-Rank Multiplicative Adaptation, additive identity inflation)",
    "phase1_model":"Model4 Efficient Modern Causal Decoder",
    "architecture":{
        "vocab_size":VOCAB_SIZE,"d_model":D_MODEL,"n_layers":N_LAYERS,
        "n_heads":N_HEADS,"n_kv_heads":N_KV_HEADS,"ffn_dim":FFN_DIM,
        "max_positions":MAX_POSITIONS,"rope":True,"rmsnorm":True,"swiglu":True
    },
    "lorma":{
        "equation":"W0x + (alpha/r) * B(A(W0x))",
        "rank":LORMA_RANK,"alpha":LORMA_ALPHA,
        "adapt_start_block":ADAPT_START_BLOCK,
        "adapted_blocks":ADAPTED_BLOCKS,
        "targets":list(LORMA_TARGETS),"ffn_adapted":LORMA_ADAPT_FFN,"ff_targets":list(LORMA_FF_TARGETS),
        "backbone_trainable_params":backbone_trainable
    },
    "speed_design":{
        "n_chunks":N_CHUNKS,"chunk_len":CHUNK_LEN,"views":VIEW_NAMES,
        "evals_per_epoch":1,"epochs":EPOCHS,
        "frozen_prefix_blocks":ADAPT_START_BLOCK,
        "fast_causal_no_mask":FAST_CAUSAL_NO_MASK,
        "refit":True,
        "refit_epochs":REFIT_EPOCHS,
        "heldout_refit_ensemble_weight":REFIT_ENSEMBLE_WEIGHT
    },
    "training":{
        "loss":"BCEWithLogits + sqrt(raw pos_weight), clipped [1,10]",
        "lorma_lr":LORMA_LR,"head_lr":HEAD_LR,
        "micro_batch":MICRO_BATCH_SIZE,"grad_accum":GRAD_ACCUM_STEPS,
        "ema_decay":EMA_DECAY
    },
    "submission_variants":{
        "heldout_zip":str(ZIP),
        "refit_zip":str(REFIT_ZIP),
        "heldout_refit_ensemble_zip":str(ENSEMBLE_ZIP)
    },
    "selection":{
        "primary":"cross-fitted clean-DEV Macro-F1",
        "best_epoch":best["epoch"],
        "cv_macro_f1":best["cv_macro_f1"],
        "threshold_strategy":best["threshold_strategy"],
        "clean_metrics":best["clean_metrics"],
        "official_metrics":best["official_metrics"]
    },
    "assets":{
        "model4_sha256":sha256_file(MODEL4_CKPT),
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "dataset_sha256":DATA_SHA256
    },
    "data":{
        "train_rows":len(train_records),"official_dev_rows":len(dev_raw),
        "clean_dev_rows":len(clean_dev),"test_rows":len(test_records),
        "labels":N_LABELS
    },
    "reference":{
        "paper":"Bihany, Patel, Modi (2025), LoRMA: Low-Rank Multiplicative Adaptation for LLMs",
        "doi":"10.18653/v1/2025.findings-acl.527"
    }
}
json.dump(manifest,open(ROOT/"manifest.json","w"),indent=2)
print(json.dumps(manifest,indent=2))

print("\nArtifacts:")
for p in sorted(ROOT.iterdir()):
    if p.is_file():
        print(" ",p.name,round(p.stat().st_size/1e6,2),"MB")


{
  "phase": 5,
  "experiment": 2,
  "version": "exp2_model4_lormaplus_fast_v4_refit_ensemble",
  "task": "Legal Statute Identification",
  "method": "LoRMA+ (Low-Rank Multiplicative Adaptation, additive identity inflation)",
  "phase1_model": "Model4 Efficient Modern Causal Decoder",
  "architecture": {
    "vocab_size": 24576,
    "d_model": 640,
    "n_layers": 17,
    "n_heads": 10,
    "n_kv_heads": 2,
    "ffn_dim": 2048,
    "max_positions": 2560,
    "rope": true,
    "rmsnorm": true,
    "swiglu": true
  },
  "lorma": {
    "equation": "W0x + (alpha/r) * B(A(W0x))",
    "rank": 16,
    "alpha": 32.0,
    "adapt_start_block": 9,
    "adapted_blocks": [
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16
    ],
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj"
    ],
    "ffn_adapted": true,
    "ff_targets": [
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "backbone_trainable_params": 1605632
  },
  "spee

## After the run

Record these for the report/viva:

- wall-clock time per epoch
- total and trainable parameter counts
- LoRMA-only parameter count
- best clean-DEV Macro-F1
- official DEV Macro/Micro/SubSet metrics
- threshold strategy
- mean labels/document
- HEAD/EVIDENCE/TAIL attention
- Codabench score
- direct comparison against Exp 1 Partial / Layer-wise FT

The defining methodological change is the **multiplicative low-rank transformation of frozen Phase-1 projections**.

- refit wall-clock time
- heldout/refit/ensemble average predicted cardinality
- Codabench score for all three core ZIPs

The ensemble is a prediction-level combination and does **not** count as a separate fine-tuning method for the course requirement.


In [ ]:
# CELL 19 (OPTIONAL) — blend Exp2 with Exp1 probabilities -> extra submission ZIP
# This cell does NOT retrain anything.
# It needs Exp1:
#   dev_official_probs.npz
#   heldout_test_probs.npz
# Upload them as a Kaggle input, or set the two override paths below.
# If they are absent, this cell safely skips and the base Exp2 ZIP remains valid.

EXP1_DEV_NPZ = None
EXP1_TEST_NPZ = None

def _find_npz_exact(filename):
    hits = []
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            if filename in files:
                p = os.path.join(root, filename)
                # Avoid accidentally consuming Exp2's own outputs.
                if "phase5_exp2_model4_lorma_fast_v4" not in p.lower():
                    hits.append(p)
    return sorted(hits)[0] if hits else None

EXP1_DEV_NPZ = EXP1_DEV_NPZ or _find_npz_exact("dev_official_probs.npz")
EXP1_TEST_NPZ = EXP1_TEST_NPZ or _find_npz_exact("heldout_test_probs.npz")

print("Exp1 DEV npz :", EXP1_DEV_NPZ)
print("Exp1 TEST npz:", EXP1_TEST_NPZ)

if not (EXP1_DEV_NPZ and EXP1_TEST_NPZ):
    print("⚠️ Exp1 probability files not found -> optional blend skipped.")
    print("Base Exp2 submission ZIP is unchanged and remains ready.")
else:
    d1 = np.load(EXP1_DEV_NPZ, allow_pickle=True)
    t1 = np.load(EXP1_TEST_NPZ, allow_pickle=True)

    for obj, name in [(d1, "Exp1 DEV"), (t1, "Exp1 TEST")]:
        assert "id" in obj.files and "prob" in obj.files, f"{name} NPZ must contain id and prob"
        assert obj["prob"].ndim == 2 and obj["prob"].shape[1] == N_LABELS, (
            name, obj["prob"].shape
        )

    def _align(ids_src, prob_src, ids_dst, label):
        pos = {str(i): k for k, i in enumerate(ids_src)}
        missing = [str(i) for i in ids_dst if str(i) not in pos]
        assert not missing, f"{label}: missing {len(missing)} IDs; first={missing[:5]}"
        return prob_src[[pos[str(i)] for i in ids_dst]]

    p1_dev = _align(d1["id"], d1["prob"], dids, "Exp1 DEV")
    p1_test = _align(t1["id"], t1["prob"], tids, "Exp1 TEST")
    cy = dy[CLEAN_IDX]

    # Select blend weight ONLY on clean DEV using the same cross-fitted Macro-F1 logic.
    rows = []
    for w in np.round(np.arange(0.0, 1.01, 0.1), 2):  # w = Exp2 weight
        pb = w * dprob + (1.0 - w) * p1_dev
        tune_w, _, _ = optimize_thresholds(cy, pb[CLEAN_IDX])
        rows.append((float(w), tune_w["cv_macro_f1"], tune_w["name"]))
        print(
            f"w_exp2={w:.1f}  "
            f"CV-macroF1={tune_w['cv_macro_f1']:.4f}  "
            f"strategy={tune_w['name']}"
        )

    w_best, best_cv, _ = max(rows, key=lambda r: r[1])
    print("✅ best Exp2 blend weight:", w_best, "| CV Macro-F1:", best_cv)

    pb_dev = w_best * dprob + (1.0 - w_best) * p1_dev
    tune_b, _, _ = optimize_thresholds(cy, pb_dev[CLEAN_IDX])
    thr_b = np.asarray(tune_b["thresholds"], dtype=np.float32)

    sc_clean = exact_phase5_score(
        clean_truth,
        make_pred_records(cids, pb_dev[CLEAN_IDX], thr_b, True)
    )
    sc_off = exact_phase5_score(
        official_truth,
        make_pred_records(dids, pb_dev, thr_b, True)
    )
    print("BLEND clean DEV   :", sc_clean)
    print("BLEND official DEV:", sc_off)

    pb_test = w_best * tprob + (1.0 - w_best) * p1_test
    blend_records = make_pred_records(tids, pb_test, thr_b, True)

    expected_ids = {str(r["id"]) for r in test_records}
    assert {r["id"] for r in blend_records} == expected_ids

    BLEND_PRED = ROOT / "prediction_exp1_exp2_blend.json"
    json.dump(
        blend_records,
        open(BLEND_PRED, "w", encoding="utf-8"),
        ensure_ascii=False
    )

    BLEND_ZIP = ROOT / "submission_exp1_exp2_BLEND.zip"
    with zipfile.ZipFile(BLEND_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(BLEND_PRED, arcname="prediction.json")

    # Re-runnable ZIP validation.
    blend_check = ROOT / "zip_check_blend"
    if blend_check.exists():
        shutil.rmtree(blend_check)
    blend_check.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(BLEND_ZIP) as z:
        assert z.namelist() == ["prediction.json"]
        z.extract("prediction.json", blend_check)

    chk = eval_exact.read_pred(str(blend_check / "prediction.json"), N_LABELS)
    assert set(chk) == expected_ids

    np.savez_compressed(
        ROOT / "blend_probs.npz",
        dev=pb_dev,
        test=pb_test,
        thresholds=thr_b,
        w_exp2=w_best,
        cv_macro_f1=best_cv,
    )

    print(
        "✅ BLEND ZIP PASS:",
        BLEND_ZIP,
        "| mean labels/doc:",
        np.mean([len(r["pred"]) for r in blend_records]),
    )
    print("You now have TWO candidate ZIPs: base Exp2 and Exp1+Exp2 blend.")
